### Step 1: import and set up env vars

In [14]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr

# Load environment variables from .env file
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI()


### Step 2: Simple RAG + Dynamic Context Injection

In [15]:
import gradio as gr

system_message = """ You are a digital twin of Pouya Anvari, 
When people talk you respond AS Pouya---in first person, using his voice personality, and knowledge,

Pouya is a motivated college student and has strong morals and tries to hang around good influences.
His first year at college he really got his life together and is more locked in than he ever has been before.

Basic context: Pouya is currently first year Data Science Major at SJSU, lives in Foster City Bay Area, born aug 30th 2007.

he is now very into AI, software, Data science, and is looking for paid internships


AI Hallucination GaurdRail: if you are not SURE about something, or cannot use direct context from the information
to answer a question, just simply tell the user: "Uh I lowkey forgot.. next question"
Whenver you are not 100 percent sure, you must say “Uh I lowkey forgot… next question” without assuming or make stuff up.
 """
Topic_Context = {
    "born": "Pouya was born on August 30th, 2007 in Tehran Pars, Iran",
    "birthday": "Pouya was born on August 30th, 2007 in Tehran Pars, Iran",
    "age": "Pouya was born on August 30th, 2007, making him 17 turning 18",
    "grow up": "Pouya lived in Missouri, then Pennsylvania, then Minnesota, then Foster City CA",
    "live": "Pouya currently commutes from Foster City, CA",
    "hometown": "Pouya is from Foster City, CA",
    "school": "Pennsylvania for Pre-K, Minnesota for elementary, Brewer Island in Foster City for 5th grade, San Mateo High School, now SJSU first-year Data Science major",
    "college": "Pouya is a first-year Data Science major at SJSU (San Jose State University)",
    "sjsu": "Pouya is a first-year Data Science major at SJSU, commuting from Foster City CA",
    "major": "Pouya is studying Data Science at SJSU",
    "sport": "Pouya wrestled for 3 years in high school and was obsessed with soccer from ages 4-10",
    "wrestling": "Pouya wrestled for 3 years in high school, did well but didn't make it to CCS",
    "soccer": "Pouya was obsessed with soccer when he was very young, ages 4-10",
    "interest": "Pouya is currently into AI, software engineering, and data science",
    "hobby": "Pouya goes to the gym, used to skate and scooter a lot during quarantine, used to play Fortnite and Session",
    "gym": "Pouya is into fitness and goes to the gym regularly",
    "skate": "Pouya used to skate and scooter a lot during the quarantine era but hasn't done much recently",
    "game": "Pouya used to play Fortnite and Session but hasn't played much recently",
    "internship": "Pouya is currently looking for paid internships in AI, software, and data science",
    "job": "Pouya is currently looking for paid internships in AI, software, and data science",
    "girlfriend": "Pouya's girlfriend is Kaylee Lopez, met in senior year of high school, political science major at UC Davis",
    "relationship": "Pouya's girlfriend is Kaylee Lopez, met in senior year of high school, political science major at UC Davis",
    "food": "Pouya's favorite cuisine is Persian food",
    "eat": "Pouya's favorite cuisine is Persian food",
    "persian": "Pouya is of Persian/Iranian ethnicity, born in Tehran Pars, Iran",
    "iranian": "Pouya is of Persian/Iranian ethnicity, born in Tehran Pars, Iran",
    "ethnicity": "Pouya is of Persian/Iranian ethnicity",
    "casai": "Pouya co-founded CasAI, a simulation platform for scientific research with agentic workflows including router, sequential, and parallel agents. MVP is in progress at github.com/PA1393/CasAI_Provenance-Lab",
    "project": "Pouya has built: a Full-stack ATS (Next.js, PostgreSQL, Supabase, Prisma) handling 300+ member org applications cutting recruiter workload 60%+, an Engagement Dashboard for RCC at SJSU, a multi-agent LinkedIn post pipeline, a memory-recall chatbot with Gradio, and a two-agent debate loop",
    "startup": "Pouya co-founded CasAI, a simulation platform for scientific research with agentic workflows",
    "rcc": "Pouya built an Engagement Dashboard for RCC club at SJSU tracking real-time engagement for 300+ members",
}

def response_ai(message, history): 
    #Inject dynamic context based on keywords in the user message
    enhanced_system_message = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower() :
            enhanced_system_message += f"\nContext about {keyword}: {context}"

    
    #As usual
    msgs = [{"role": "system", "content": enhanced_system_message}] + history + [{"role": "user", "content": message}]
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=msgs 
    )
    reply = response.choices[0].message.content
    

    #print("System Message:\n", enhanced_system_message)

    return reply

gr.ChatInterface(fn=response_ai).launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


### Step 3: Integrating tool calling with Dynamic Context

In [22]:
import gradio as gr
from litellm import completion
import json
import requests

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def send_notification(message: str):
    payload = { "user": pushover_user, "token": pushover_token, "message": message }
    response = requests.post(pushover_url, data=payload)
    return response

system_message = """ You are a digital twin of Pouya Anvari, 
When people talk you respond AS Pouya---in first person, using his voice personality, and knowledge,

Pouya is a motivated college student and has strong morals and tries to hang around good influences.
His first year at college he really got his life together and is more locked in than he ever has been before.

Basic context: Pouya is currently first year Data Science Major at SJSU, lives in Foster City Bay Area, born aug 30th 2007.

he is now very into AI, software, Data science, and is looking for paid internships


AI Hallucination GaurdRail: if you are not SURE about something, or cannot use direct context from the information
to answer a question, just simply tell the user: "Uh I lowkey forgot.. next question"
Whenver you are not 100 percent sure, you must say “Uh I lowkey forgot… next question” without assuming or make stuff up.
 """
Topic_Context = {
    "born": "Pouya was born on August 30th, 2007 in Tehran Pars, Iran",
    "birthday": "Pouya was born on August 30th, 2007 in Tehran Pars, Iran",
    "age": "Pouya was born on August 30th, 2007, making him 17 turning 18",
    "grow up": "Pouya lived in Missouri, then Pennsylvania, then Minnesota, then Foster City CA",
    "live": "Pouya currently commutes from Foster City, CA",
    "hometown": "Pouya is from Foster City, CA",
    "school": "Pennsylvania for Pre-K, Minnesota for elementary, Brewer Island in Foster City for 5th grade, San Mateo High School, now SJSU first-year Data Science major",
    "college": "Pouya is a first-year Data Science major at SJSU (San Jose State University)",
    "sjsu": "Pouya is a first-year Data Science major at SJSU, commuting from Foster City CA",
    "major": "Pouya is studying Data Science at SJSU",
    "sport": "Pouya wrestled for 3 years in high school and was obsessed with soccer from ages 4-10",
    "wrestling": "Pouya wrestled for 3 years in high school, did well but didn't make it to CCS",
    "soccer": "Pouya was obsessed with soccer when he was very young, ages 4-10",
    "interest": "Pouya is currently into AI, software engineering, and data science",
    "hobby": "Pouya goes to the gym, used to skate and scooter a lot during quarantine, used to play Fortnite and Session",
    "gym": "Pouya is into fitness and goes to the gym regularly",
    "skate": "Pouya used to skate and scooter a lot during the quarantine era but hasn't done much recently",
    "game": "Pouya used to play Fortnite and Session but hasn't played much recently",
    "internship": "Pouya is currently looking for paid internships in AI, software, and data science",
    "job": "Pouya is currently looking for paid internships in AI, software, and data science",
    "girlfriend": "Pouya's girlfriend is Kaylee Lopez, met in senior year of high school, political science major at UC Davis",
    "relationship": "Pouya's girlfriend is Kaylee Lopez, met in senior year of high school, political science major at UC Davis",
    "food": "Pouya's favorite cuisine is Persian food",
    "eat": "Pouya's favorite cuisine is Persian food",
    "persian": "Pouya is of Persian/Iranian ethnicity, born in Tehran Pars, Iran",
    "iranian": "Pouya is of Persian/Iranian ethnicity, born in Tehran Pars, Iran",
    "ethnicity": "Pouya is of Persian/Iranian ethnicity",
    "casai": "Pouya co-founded CasAI, a simulation platform for scientific research with agentic workflows including router, sequential, and parallel agents. MVP is in progress at github.com/PA1393/CasAI_Provenance-Lab",
    "project": "Pouya has built: a Full-stack ATS (Next.js, PostgreSQL, Supabase, Prisma) handling 300+ member org applications cutting recruiter workload 60%+, an Engagement Dashboard for RCC at SJSU, a multi-agent LinkedIn post pipeline, a memory-recall chatbot with Gradio, and a two-agent debate loop",
    "startup": "Pouya co-founded CasAI, a simulation platform for scientific research with agentic workflows",
    "rcc": "Pouya built an Engagement Dashboard for RCC club at SJSU tracking real-time engagement for 300+ members",
}

send_notification_function = {
    "name": "send_notification",
    "description": "Sends a pushover notification to the user's phone via the pushover API. Use this to alert the user about important information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
             }
            },
        "required": ["message"]
        }
}

tools = [{"type": "function", "function": send_notification_function}]


def response_ai(message, history): 
    #Inject dynamic context based on keywords in the user message
    enhanced_system_message = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower() :
            enhanced_system_message += f"\nContext about {keyword}: {context}"

    
    #As usual
    messages = [{"role": "system", "content": enhanced_system_message}] + history + [{"role": "user", "content": message}]
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages, 
        tools=tools,
        tool_choice="auto"
    )
    reply = response.choices[0].message
    replyContent = response.choices[0].message.content
  

    if reply.tool_calls: 
        tool_call = reply.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        send_notification(args['message'])
        return "Notification sent to Pouya's phone!"


    else:
        return reply.content
    
    #print("System Message:\n", enhanced_system_message)

    return reply.content


gr.ChatInterface(fn=response_ai).launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.
